# equity-forecast — results walkthroughThis notebook **contains no modelling logic**. It loads the artefacts a run wrote to`results/<run_id>/` and puts them side by side. All of the science lives in the package(`src/equity_forecast/`) and is covered by the test suite; a notebook is the wrong place foranything you would want to be reproducible.To produce a run:```bashequity-forecast run --config configs/default.yaml```

In [ ]:
from pathlib import Pathimport matplotlib.pyplot as pltimport pandas as pdpd.set_option("display.width", 160)pd.set_option("display.max_columns", 50)pd.set_option("display.float_format", lambda v: f"{v:,.4f}")RESULTS = Path("../results")def latest_run(results_dir: Path = RESULTS) -> Path:    """Return the most recently modified run directory, ignoring `latest/`."""    runs = [p for p in results_dir.iterdir() if p.is_dir() and p.name != "latest"]    if not runs:        raise FileNotFoundError(            f"no runs found under {results_dir.resolve()} - "            "run `equity-forecast run --config configs/default.yaml` first"        )    return max(runs, key=lambda p: p.stat().st_mtime)RUN = latest_run()print("Reading:", RUN.resolve())sorted(p.name for p in RUN.iterdir())

## 1. Headline: every model against the random-walk baselineThe only number that matters is whether a model beats `zero_return` — and whether that margin survives a significance test.

In [ ]:
summary = pd.read_csv(RUN / "metrics_summary.csv")summary

In [ ]:
print((RUN / "summary.md").read_text())

## 2. Dispersion across foldsA single mean hides everything. Walk-forward gives one observation per fold, so look at the spread.

In [ ]:
by_fold = pd.read_csv(RUN / "metrics_by_fold.csv")pooled = by_fold[by_fold["ticker"] == "ALL"]fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))for ax, metric, ref in ((axes[0], "dir_acc", 0.5), (axes[1], "ic", 0.0)):    groups = [g[metric].dropna().to_numpy() for _, g in pooled.groupby("model")]    labels = [name for name, _ in pooled.groupby("model")]    ax.boxplot(groups, tick_labels=labels)    ax.axhline(ref, ls="--", c="crimson", lw=1)    ax.set_title(f"per-fold {metric}")    ax.tick_params(axis="x", rotation=30)    ax.grid(alpha=0.3)fig.tight_layout()

## 3. Does any of it survive transaction costs?

In [ ]:
backtest = pd.read_csv(RUN / "backtest_summary.csv")backtest

In [ ]:
pnl = pd.read_parquet(RUN / "pnl.parquet")wide = pnl.pivot_table(index="date", columns="model", values="net").sort_index()ax = wide.cumsum().plot(figsize=(12, 5), lw=1.4)ax.set_title("Cumulative net P&L per model (equal-weight, after costs)")ax.set_ylabel("cumulative log-return contribution")ax.axhline(0.0, c="black", lw=0.8)ax.grid(alpha=0.3)

## 4. Forecast diagnosticsIf a model has any edge it should show up as a positive relationship between forecast andrealised return. Bin the forecasts and look.

In [ ]:
forecasts = pd.read_parquet(RUN / "forecasts.parquet")model = "ridge" if "ridge" in set(forecasts["model"]) else sorted(set(forecasts["model"]))[0]sub = forecasts[forecasts["model"] == model].dropna(subset=["forecast", "y_true"])bins = pd.qcut(sub["forecast"], 10, labels=False, duplicates="drop")profile = sub.groupby(bins)["y_true"].agg(["mean", "count"])ax = (profile["mean"] * 1e4).plot(kind="bar", figsize=(10, 4), color="steelblue")ax.set_title(f"Mean realised next-day return by forecast decile - {model}")ax.set_xlabel("forecast decile (0 = most negative)")ax.set_ylabel("mean realised return (bps)")ax.axhline(0.0, c="black", lw=0.8)ax.grid(alpha=0.3, axis="y")profile

## 5. Walk-forward geometryA sanity check that the folds are what you think they are: train blocks must end strictlybefore test blocks begin, with the embargo in between.

In [ ]:
folds = pd.read_csv(RUN / "folds.csv", parse_dates=["train_start", "train_end", "test_start", "test_end"])assert (folds["train_end"] < folds["test_start"]).all(), "look-ahead in the fold geometry"fig, ax = plt.subplots(figsize=(11, 0.42 * len(folds) + 1.5))for _, row in folds.iterrows():    y = row["fold_id"]    ax.barh(y, row["train_end"] - row["train_start"], left=row["train_start"], color="steelblue", height=0.6)    ax.barh(y, row["test_end"] - row["test_start"], left=row["test_start"], color="darkorange", height=0.6)ax.set_yticks(folds["fold_id"])ax.set_ylabel("fold")ax.set_title("Walk-forward folds (blue = train, orange = out-of-sample test)")ax.grid(alpha=0.3, axis="x")fig.tight_layout()folds